# CoT Synthesis - PER-DATASET (Kaggle GPU) — teacher **Gemma-2**

Notebook ini PEMBANDING generator: teknis sama persis dengan `cot_pipeline_kaggle.ipynb` (per-dataset), **kecuali teacher = Gemma-2** (backend `hf`/transformers, karena vLLM gak bisa jalanin Gemma-2 di T4 — `head_dim=256`). Judge tetap vLLM.

3 dataset terpisah (sudah di-clean teman, ada di `data/Final/` di repo):

```
data/Final/{numglue,aimo_hard,easy}_clean.jsonl
   -> generate (teacher Gemma-2 hf, N solusi/soal)  cot/<ds>/cot_<ds>_<teacher>.jsonl
   -> filter   (judge vLLM) + emit ChatML            sft/<ds>/{cot,nocot}.jsonl
```

Tiap dataset diproses **terpisah** -> output CoT sendiri per folder (`cot/numglue/`, `cot/aimo_hard/`, `cot/easy/`). Nama file ikut teacher (`cot_aimo_hard_gemma-2-2b-it.jsonl`) -> ganti teacher = file baru, gak nimpa.

**RUN 1-per-1:** di cell Config ada `RUN = [...]` -> pilih SATU dataset, run generate+filter, kelar -> ganti dataset berikutnya -> run lagi. Mau sekaligus 3? `RUN = DATASETS`.

Filter langsung menghasilkan ChatML (1 solusi terbaik/soal, reasoning Indonesia saja).

**Prasyarat Kaggle:**
1. Settings -> Accelerator = **GPU T4 x2**.
2. Settings -> Internet = **ON** (clone repo + download model HF).
3. Add-ons -> Secrets -> tambah `HF_TOKEN` (Gemma model gated, butuh login HF).
4. Data ikut clone repo (`data/Final/`), **gak perlu** upload Kaggle Dataset.

Full makan waktu lama. Coba dulu `LIMIT` kecil. Full: `LIMIT=None`. Resumable: kalau sesi putus, jalankan lagi (skip soal yang sudah `N` kandidat per file).

## 0. Setup (install + clone repo)

In [ ]:
!pip install -q vllm math-verify
# T4 (sm_75): FlashInfer attention bisa di-compile tapi GAGAL saat jalan ("BatchPrefill invalid argument").
# Buang flashinfer -> vLLM auto fallback ke Triton attention (jalan di T4, tanpa JIT/libcuda).
!pip uninstall -y -q flashinfer-python flashinfer 2>/dev/null; echo "flashinfer dibuang (pakai Triton attention)"

_Catatan: flashinfer sudah dibuang di cell install, jadi vLLM pakai Triton attention — tidak ada JIT FlashInfer, jadi fix libcuda tidak diperlukan lagi. (Kalau suatu saat pakai flashinfer di GPU lain, baru perlu symlink `libcuda.so`.)_

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO = '/kaggle/working/FP_NLP'
URL  = 'https://github.com/henray404/FP_NLP.git'  # private -> kalau gagal, lihat cell catatan di bawah

if os.path.exists(REPO):
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)   # selalu ambil kode terbaru
else:
    subprocess.run(['git', 'clone', '-q', URL, REPO], check=True)

sys.path.insert(0, REPO)
WORK = Path('/kaggle/working')
print('repo siap (terbaru):', REPO)
print('CATATAN: kalau ubah src/, restart kernel setelah pull supaya modul lama tidak nyangkut di memori.')

In [ ]:
# HF login via Kaggle Secret (nama secret HARUS persis "HF_TOKEN").
# Add-ons -> Secrets -> tambah HF_TOKEN, lalu attach ke notebook ini.
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

tok = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = tok
os.environ["HUGGING_FACE_HUB_TOKEN"] = tok   # vLLM/transformers baca var ini juga
login(token=tok)
print("HF login OK")

## 1. Data (3 dataset dari repo `data/Final/`)

In [ ]:
DATA_DIR = Path(REPO) / 'data' / 'Final'
DATASETS = ['numglue_clean', 'aimo_hard_clean', 'easy_clean']

for name in DATASETS:
    p = DATA_DIR / f'{name}.jsonl'
    assert p.exists(), f'tidak ketemu {p} -- git pull main terbaru di cell Setup'
    n = sum(1 for line in open(p, encoding='utf-8') if line.strip())
    print(f' - {name}: {n} soal  ({p})')

## 2. Sanity check (sudah clean + ada jawaban)

File `data/Final/` sudah di-dedup + di-clean teman (disjoint antar-dataset, semua punya `jawaban`). Cell ini cuma verifikasi, bukan dedup ulang.

In [ ]:
import json

for name in DATASETS:
    rows = [json.loads(l) for l in open(DATA_DIR / f'{name}.jsonl', encoding='utf-8') if l.strip()]
    no_jaw = sum(1 for r in rows if not (r.get('jawaban') or '').strip())
    src = sorted({r.get('source', '') for r in rows})
    print(f'{name}: {len(rows)} soal | tanpa jawaban: {no_jaw} | source={src}')

## 3. Config

In [ ]:
# Teknis SAMA dengan cot_pipeline_kaggle.ipynb, KECUALI teacher = Gemma-2.
# Gemma-2 generate pakai backend 'hf' (transformers lokal): vLLM gak bisa jalanin Gemma-2 di T4
# (head_dim=256 -> Triton OutOfResources, engine V0 dihapus). Bukan API -> no rate limit.
# Judge tetap vLLM (Qwen head_dim=128, aman di T4).
TEACHER_MODEL = 'google/gemma-2-2b-it'      # teacher generator (yg dibandingkan)
JUDGE_MODEL   = 'Qwen/Qwen2.5-7B-Instruct'  # judge

N_CANDIDATES = 4       # solusi per soal
MAX_TOKENS   = 3072    # samain dgn cot_pipeline_kaggle (reasoning panjang; lokal jadi gak ada limit)
TP           = 2       # 2x T4 -> tensor parallel 2 (dipakai judge/filter vLLM)
LIMIT        = None    # cap soal BARU per dataset (COBA dulu kecil). Full: None

# >>> RUN 1-PER-1: pilih dataset yang diproses run INI (kebal quota, gampang dipantau).
# Ganti ke salah satu, run generate+filter, kelar -> ganti dataset berikutnya -> run lagi.
# Mau sekaligus 3? isi RUN = DATASETS.
RUN = ['aimo_hard_clean']    # <- ganti: 'numglue_clean' / 'aimo_hard_clean' / 'easy_clean'

TEACHER_TAG = TEACHER_MODEL.split('/')[-1]   # masuk nama file output -> ganti teacher = file baru
COT_DIR = WORK / 'cot'
SFT_DIR = WORK / 'sft'

def ds_short(name):                     # 'aimo_hard_clean' -> 'aimo_hard'
    return name.replace('_clean', '')

def cand_path(name):                    # cot/<ds>/cot_<ds>_<teacher>.jsonl
    d = COT_DIR / ds_short(name); d.mkdir(parents=True, exist_ok=True)
    return d / f'cot_{ds_short(name)}_{TEACHER_TAG}.jsonl'

def correct_path(name):                 # cot/<ds>/correct_<ds>_<teacher>.jsonl
    d = COT_DIR / ds_short(name); d.mkdir(parents=True, exist_ok=True)
    return d / f'correct_{ds_short(name)}_{TEACHER_TAG}.jsonl'

def chatml_dir(name):                   # sft/<ds>/{cot,nocot}.jsonl
    return SFT_DIR / ds_short(name)

# Shard (bagi soal disjoint sama teman). Solo: NUM_SHARDS=1. Berdua: kamu 0, teman 1.
NUM_SHARDS = 1
SHARD_I    = 0

assert all(r in DATASETS for r in RUN), f'RUN harus subset {DATASETS}, dapat {RUN}'
print('teacher:', TEACHER_MODEL, '| tag:', TEACHER_TAG)
print('RUN (run ini):', RUN, '| N:', N_CANDIDATES, '| limit:', LIMIT)
for name in RUN:
    print('  ->', cand_path(name))

## 3b. Resume antar-session (checkpoint per dataset)

Biar bisa **stop session** tanpa ngulang dari 0:

1. **Paling gampang** — Notebook *Settings* -> **Persistence** -> `Files only`. `/kaggle/working` selamat waktu stop, jadi tiap `cot/<ds>/cot_*.jsonl` tetap ada -> run berikut auto-lanjut.
2. **Backup** (kalau persistence off): zip output (cell terakhir) -> download -> upload sebagai Kaggle Dataset -> **Add**, lalu copy manual ke `cot/`.

Yang lanjut otomatis: generate skip soal yang sudah `N` kandidat (per file); filter skip kandidat yang sudah dinilai (`*.progress`).

In [ ]:
COT_DIR.mkdir(parents=True, exist_ok=True)

# Status checkpoint untuk dataset di RUN (generate auto-skip soal yang sudah N kandidat).
for name in RUN:
    cp = cand_path(name)
    if cp.exists():
        n = sum(1 for _ in open(cp, encoding='utf-8'))
        print(f'{name}: {n} kandidat sudah ada -> generate lanjut dari sini')
    else:
        print(f'{name}: belum ada -> generate mulai dari 0')
    prog = Path(str(correct_path(name)) + '.progress')
    if prog.exists():
        print(f'   judge progress: {sum(1 for _ in open(prog, encoding="utf-8"))} kandidat sudah dinilai')

## 4. Generate (teacher Gemma-2, transformers lokal)

Notebook ini bandingin generator **Gemma-2** (vs DeepSeek-R1 di `cot_pipeline_kaggle.ipynb`).

**Kenapa transformers, bukan vLLM (dan bukan API):** vLLM gak bisa jalanin Gemma-2 di T4 — `head_dim=256`
bikin kernel Triton attention minta shared memory > limit T4 (sm75 64KB) -> `OutOfResources` /
`EngineDeadError`, dan vLLM baru sudah buang engine V0 jadi workaround env diabaikan. Backend `hf`
(transformers, `attn_implementation="eager"`) jalan lokal tanpa limit itu — **bukan API, no rate
limit**. Gemma-2-2b (2B) muat santai di 1x T4. Lebih lambat dari vLLM tapi stabil; tulis incremental
jadi aman ke-timeout. Resumable: jalankan lagi, soal yang sudah `N` kandidat dilewati.

In [ ]:
import subprocess
# libcuda symlink (jaga-jaga buat lib lain). Generate pakai backend transformers, jadi tidak
# butuh env attention vLLM apa pun.
libs = subprocess.run("find / -name 'libcuda.so*' 2>/dev/null",
                    shell=True, capture_output=True, text=True).stdout.split()
src = next((l for l in libs if l.endswith('libcuda.so.1')), libs[0] if libs else '')
for d in ['/usr/local/cuda/lib64', '/usr/local/cuda/lib64/stubs']:
    subprocess.run(f'mkdir -p {d} && ln -sf {src} {d}/libcuda.so', shell=True)
subprocess.run('echo /usr/local/cuda/lib64 > /etc/ld.so.conf.d/zzz_cuda.conf && ldconfig',
shell=True)
print('libcuda.so ->', src)

In [ ]:
from src.cot_synthesis.generate_gemma import run_generate

# backend='hf': transformers lokal (eager attention) — vLLM gak dipakai DI SINI karena crash buat
# Gemma-2 di T4 (head_dim=256). Bukan API -> no rate limit. Gemma-2-2b muat 1x T4. Resumable per file.
# Generate tiap dataset di RUN -> cot/<ds>/cot_<ds>_<teacher>.jsonl.
for name in RUN:
    src_path = DATA_DIR / f'{name}.jsonl'
    out_path = cand_path(name)
    print('=' * 12, 'generate', name, '->', out_path)
    stats = run_generate(
        src_path, out_path, backend='hf', model=TEACHER_MODEL,
        n=N_CANDIDATES, max_tokens=MAX_TOKENS, limit=LIMIT,
        shard=SHARD_I, num_shards=NUM_SHARDS,
    )
    print(name, stats)

## 5. Bebaskan GPU sebelum load judge

Kalau cell filter di bawah **OOM**: Restart kernel, lalu jalankan ulang cell Setup -> Config -> Filter (file `candidates.jsonl` di /kaggle/working tetap ada).

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()
print('GPU dibersihkan')

## 7. (Opsional) Bangun ulang ChatML

Cell Filter di atas **sudah** menghasilkan `sft/<ds>/cot.jsonl` + `nocot.jsonl` per dataset (emit_chatml=True). Jalankan cell ini hanya kalau mau ganti policy (mis. simpan semua solusi/soal, atau izinkan Inggris).

## 7. (Opsional) Bangun ulang ChatML

Cell Filter di atas **sudah** menghasilkan `sft/cot.jsonl` + `nocot.jsonl` (emit_chatml=True).
Jalankan cell ini hanya kalau mau ganti policy (mis. simpan semua solusi/soal, atau izinkan Inggris).

In [ ]:
# Opsional: bangun ulang ChatML untuk dataset di RUN dari correct_*.jsonl dengan policy berbeda
# (mis. simpan semua solusi/soal, atau izinkan Inggris).
from src.cot_synthesis.to_chatml import run as build_chatml

for name in RUN:
    corr = correct_path(name)
    if not corr.exists():
        print('skip:', name); continue
    ml = build_chatml(corr, chatml_dir(name), best_per_problem=True, id_only=True,
                      max_solution_chars=8000)
    print(name, ml)

## 8. Download hasil

Output ada di `/kaggle/working/sft/` dan `/kaggle/working/cot/`. Zip biar gampang diunduh, atau pakai **Save Version** biar persist antar sesi.

In [ ]:
import shutil
from IPython.display import FileLink

out_zip = '/kaggle/working/cot_sft_outputs'
# kumpulin cot/ + sft/ ke satu folder lalu zip
bundle = WORK / 'bundle'
bundle.mkdir(exist_ok=True)
for d in ['cot', 'sft']:
    src = WORK / d
    if src.exists():
        shutil.copytree(src, bundle / d, dirs_exist_ok=True)
shutil.make_archive(out_zip, 'zip', str(bundle))
print('zip:', out_zip + '.zip')
FileLink('cot_sft_outputs.zip')